# XLM-R base fine-tuning on ViGoEmotions Ekman-7

In [ ]:
import os, json, ast, random
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score

In [ ]:
CFG = {
    "model_name": "xlm-roberta-base",
    "num_labels": 7,
    "max_len": 200,
    "batch_size": 32,
    "epochs": 12,
    "lr": 5e-5,
    "dropout": 0.2,
    "threshold": 0.5,
    "seed": 42,
    "run_dir": "/kaggle/working/runs/xlmr_ekman7_seed42",
}
DATA_DIR = "/kaggle/input/datasets/giaphucgiaphuc/vigo-ekman7"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG["seed"])

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df   = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
print(train_df.shape, val_df.shape, test_df.shape)

In [ ]:
N = len(train_df)
label_counts = np.zeros(CFG["num_labels"])
for s in train_df["labels"]:
    for idx in ast.literal_eval(s):
        label_counts[idx] += 1

pos_weight = (N - label_counts) / label_counts
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32).to(DEVICE)
print("pos_weight:", np.round(pos_weight, 2))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])

class EmotionDataset(Dataset):
    def __init__(self, df):
        self.texts  = df["text"].tolist()
        self.labels = df["labels"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=CFG["max_len"],
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        label_indices = ast.literal_eval(self.labels[idx])
        label_vec = torch.zeros(CFG["num_labels"], dtype=torch.float32)
        for i in label_indices:
            label_vec[i] = 1.0
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         label_vec,
        }

In [ ]:
train_loader = DataLoader(EmotionDataset(train_df), batch_size=CFG["batch_size"], shuffle=True)
val_loader   = DataLoader(EmotionDataset(val_df),   batch_size=CFG["batch_size"])
test_loader  = DataLoader(EmotionDataset(test_df),  batch_size=CFG["batch_size"])

In [ ]:
class EmotionClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(CFG["model_name"])
        hidden_size     = self.encoder.config.hidden_size
        self.dropout    = nn.Dropout(CFG["dropout"])
        self.classifier = nn.Linear(hidden_size, CFG["num_labels"])

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls))

model = EmotionClassifier().to(DEVICE)
model.encoder.gradient_checkpointing_enable()

In [ ]:
optimizer     = torch.optim.AdamW(model.parameters(), lr=CFG["lr"])
total_steps   = len(train_loader) * CFG["epochs"]
scheduler     = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=len(train_loader),
    num_training_steps=total_steps,
)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

In [ ]:
os.makedirs(CFG["run_dir"], exist_ok=True)
best_macro = 0.0

for epoch in range(CFG["epochs"]):
    model.train()
    total_loss = 0
    for batch in train_loader:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["labels"].to(DEVICE)
        optimizer.zero_grad()
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = model(ids, mask)
            loss   = criterion(logits, lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for batch in val_loader:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                logits = model(ids, mask)
            preds  = (torch.sigmoid(logits) >= CFG["threshold"]).cpu().numpy()
            all_preds.append(preds)
            all_true.append(batch["labels"].numpy())

    all_preds = np.vstack(all_preds)
    all_true  = np.vstack(all_true)
    macro    = f1_score(all_true, all_preds, average="macro",    zero_division=0)
    weighted = f1_score(all_true, all_preds, average="weighted", zero_division=0)
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1:02d} | loss {avg_loss:.4f} | val macro {macro:.4f} | val weighted {weighted:.4f}")

    if macro > best_macro:
        best_macro = macro
        torch.save(model.state_dict(), os.path.join(CFG["run_dir"], "best_model.pt"))
        print("  -> saved best")

In [ ]:
model.load_state_dict(torch.load(os.path.join(CFG["run_dir"], "best_model.pt"), weights_only=True))
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for batch in test_loader:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = model(ids, mask)
        preds  = (torch.sigmoid(logits) >= CFG["threshold"]).cpu().numpy()
        all_preds.append(preds)
        all_true.append(batch["labels"].numpy())

all_preds     = np.vstack(all_preds)
all_true      = np.vstack(all_true)
test_macro    = f1_score(all_true, all_preds, average="macro",    zero_division=0)
test_weighted = f1_score(all_true, all_preds, average="weighted", zero_division=0)

print(f"Test F1-macro:    {test_macro:.4f}")
print(f"Test F1-weighted: {test_weighted:.4f}")

In [ ]:
metrics = {
    "model":            CFG["model_name"],
    "seed":             CFG["seed"],
    "best_val_macro":   best_macro,
    "test_macro":       test_macro,
    "test_weighted":    test_weighted,
}
with open(os.path.join(CFG["run_dir"], "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))